# 01. Exploratory Data Analysis (EDA)

BTC/USDT 1h OHLCV 데이터에 대한 탐색적 데이터 분석.

**목표**
- 데이터 기본 구조 및 통계량 파악
- 가격/거래량 시계열 패턴 확인
- 수익률 분포 분석
- 기술적 지표 간 상관관계 파악
- 시간대별/요일별 패턴 탐색
- 변동성 클러스터링 확인

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.data.collector import load_from_parquet
from src.data.features import build_features

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.4f}".format)

## 1. 데이터 로드 및 기본 정보

In [ ]:
df_raw = load_from_parquet("../data/raw/btc_usdt_1h_20240101_20260226.parquet")
print(f"기간: {df_raw.index.min()} ~ {df_raw.index.max()}")
print(f"행 수: {len(df_raw):,}")
print(f"컬럼: {list(df_raw.columns)}")
print()
df_raw.describe()

In [ ]:
# 결측치 및 데이터 타입 확인
print("결측치:")
print(df_raw.isna().sum())
print()
print("데이터 타입:")
print(df_raw.dtypes)

## 2. 가격/거래량 시계열 시각화

In [ ]:
# 캔들차트 + 거래량
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    vertical_spacing=0.03, row_heights=[0.7, 0.3],
    subplot_titles=("BTC/USDT 1H Price", "Volume")
)

fig.add_trace(
    go.Candlestick(
        x=df_raw.index, open=df_raw["open"], high=df_raw["high"],
        low=df_raw["low"], close=df_raw["close"], name="OHLC"
    ), row=1, col=1
)

fig.add_trace(
    go.Bar(x=df_raw.index, y=df_raw["volume"], name="Volume", marker_color="rgba(0,150,255,0.4)"),
    row=2, col=1
)

fig.update_layout(height=600, xaxis_rangeslider_visible=False, showlegend=False)
fig.show()

## 3. 기본 통계량 분석

In [ ]:
returns = df_raw["close"].pct_change().dropna()

stats = {
    "평균 수익률 (1h)": f"{returns.mean():.6f}",
    "표준편차": f"{returns.std():.6f}",
    "왜도 (Skewness)": f"{returns.skew():.4f}",
    "첨도 (Kurtosis)": f"{returns.kurtosis():.4f}",
    "최대 수익": f"{returns.max():.4%}",
    "최대 손실": f"{returns.min():.4%}",
    "양수 비율": f"{(returns > 0).mean():.2%}",
}

for k, v in stats.items():
    print(f"{k}: {v}")

## 4. 수익률 분포 분석

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("수익률 히스토그램", "Q-Q Plot (정규분포 비교)"))

# 히스토그램
fig.add_trace(
    go.Histogram(x=returns, nbinsx=200, name="Returns", marker_color="steelblue"),
    row=1, col=1
)

# Q-Q Plot
from scipy import stats as scipy_stats
sorted_returns = np.sort(returns)
theoretical_q = scipy_stats.norm.ppf(np.linspace(0.001, 0.999, len(sorted_returns)))
fig.add_trace(
    go.Scatter(x=theoretical_q, y=sorted_returns, mode="markers", marker=dict(size=1), name="Q-Q"),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=[-0.05, 0.05], y=[-0.05, 0.05], mode="lines", line=dict(dash="dash", color="red"), name="Normal"),
    row=1, col=2
)

fig.update_layout(height=400, showlegend=False)
fig.show()

## 5. 기술적 지표 상관관계 히트맵

In [ ]:
df_feat = build_features(df_raw.copy())
print(f"피처 수: {len(df_feat.columns) - 5} (OHLCV 제외)")
print(f"행 수: {len(df_feat):,}")

In [ ]:
# 주요 피처만 선택하여 상관관계
key_features = [
    "close", "volume", "sma_7", "sma_25", "ema_12",
    "macd", "rsi_14", "stoch_k", "bb_width", "atr_14",
    "obv", "mfi", "adx", "cci", "roc_10",
    "return_1", "log_return", "realized_vol_20", "volume_ratio"
]
corr = df_feat[key_features].corr()

fig = px.imshow(
    corr, text_auto=".2f", aspect="auto",
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
    title="주요 피처 상관관계 히트맵"
)
fig.update_layout(height=700, width=800)
fig.show()

## 6. 시간대별/요일별 패턴 분석

In [ ]:
df_raw["hour"] = df_raw.index.hour
df_raw["day_of_week"] = df_raw.index.dayofweek
df_raw["return_1h"] = df_raw["close"].pct_change()

fig = make_subplots(rows=1, cols=2, subplot_titles=("시간대별 평균 수익률", "요일별 평균 수익률"))

# 시간대별
hourly = df_raw.groupby("hour")["return_1h"].mean() * 100
colors_h = ["green" if v > 0 else "red" for v in hourly]
fig.add_trace(
    go.Bar(x=hourly.index, y=hourly.values, marker_color=colors_h, name="Hourly"),
    row=1, col=1
)

# 요일별 (0=월 ~ 6=일)
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
daily = df_raw.groupby("day_of_week")["return_1h"].mean() * 100
colors_d = ["green" if v > 0 else "red" for v in daily]
fig.add_trace(
    go.Bar(x=day_names, y=daily.values, marker_color=colors_d, name="Daily"),
    row=1, col=2
)

fig.update_yaxes(title_text="평균 수익률 (%)", row=1, col=1)
fig.update_yaxes(title_text="평균 수익률 (%)", row=1, col=2)
fig.update_layout(height=400, showlegend=False)
fig.show()

In [ ]:
# 시간대별 거래량 패턴
hourly_vol = df_raw.groupby("hour")["volume"].mean()

fig = go.Figure(go.Bar(
    x=hourly_vol.index, y=hourly_vol.values,
    marker_color="rgba(0,150,255,0.6)"
))
fig.update_layout(
    title="시간대별 평균 거래량",
    xaxis_title="Hour (UTC)", yaxis_title="Volume",
    height=350
)
fig.show()

## 7. 변동성 클러스터링 시각화

In [ ]:
# 롤링 변동성 (20기간)
rolling_vol = returns.rolling(20).std() * np.sqrt(24 * 365)  # 연율화

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    vertical_spacing=0.05, row_heights=[0.5, 0.5],
    subplot_titles=("BTC/USDT Close Price", "연율화 변동성 (20h Rolling)")
)

fig.add_trace(
    go.Scatter(x=df_raw.index, y=df_raw["close"], mode="lines", name="Close", line=dict(width=1)),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=rolling_vol.index, y=rolling_vol.values, mode="lines", name="Volatility",
               line=dict(width=1, color="orange"), fill="tozeroy", fillcolor="rgba(255,165,0,0.2)"),
    row=2, col=1
)

fig.update_layout(height=500, showlegend=False)
fig.show()

In [ ]:
# 수익률 자기상관 (변동성 클러스터링 확인)
abs_returns = returns.abs()
autocorr = [abs_returns.autocorr(lag=i) for i in range(1, 51)]

fig = go.Figure(go.Bar(
    x=list(range(1, 51)), y=autocorr,
    marker_color="steelblue"
))
fig.update_layout(
    title="|수익률| 자기상관 (변동성 클러스터링)",
    xaxis_title="Lag (hours)", yaxis_title="Autocorrelation",
    height=350
)
fig.show()

## 8. Summary

### 주요 발견
- **데이터**: 위 셀들의 출력 결과를 기반으로 정리
- **수익률 분포**: 정규분포 대비 fat tail (첨도 확인)
- **변동성 클러스터링**: |수익률| 자기상관이 유의미 → GARCH 계열 또는 시계열 모델 활용 근거
- **시간대/요일 패턴**: 특정 시간대/요일에 수익률/거래량 편향 존재 여부 확인
- **피처 상관관계**: 높은 상관 피처 쌍 식별 → 피처 선택 시 참고